# 03 · Regularization, Dropout and Normalization

Watch a network memorise 40 noisy points, then fix it four ways: weight decay (and why AdamW's version differs from
L2), dropout, early stopping and data augmentation. Then the layers that keep deep nets trainable: BatchNorm vs
LayerNorm, RMSNorm written from scratch (what Llama uses), pre-norm vs post-norm, and residual connections measured
as gradient flow.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · training loop, optimizers, LR schedules](02_training_loop_optimizers_lr_schedules.ipynb)

## Why this matters in interviews

- "Your training loss is great and your validation loss is terrible, what do you do?" is asked in every ML
  interview, including LLM fine-tuning ones. You need the list of fixes *and* how to tell which one is working.
- "Why LayerNorm and not BatchNorm in a transformer?" and "What is RMSNorm?" are standard architecture questions;
  being able to write RMSNorm in five lines is a strong signal.
- Residual connections and pre-norm are why 100-layer transformers train at all. Interviewers want the gradient
  argument, not just the words.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn28` | What is overfitting in the context of fine-tuning an LLM, and how do you spot it? |
| `tr09` | What does LayerNorm do, and how is RMSNorm different? |
| `tr08` | What is the residual stream, and why do residual connections matter conceptually? |
| `tr02` | Walk me through one transformer block, step by step (the norm and residual parts). |

In [ ]:
import copy
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.patches import Rectangle
from sklearn.datasets import make_moons

np.random.seed(0); torch.manual_seed(0)
torch.set_num_threads(1)                         # tiny models: single-threaded is fastest
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.8})

Xa, ya = make_moons(n_samples=40, noise=0.3, random_state=1)      # tiny, noisy training set
Xv, yv = make_moons(n_samples=1000, noise=0.3, random_state=2)    # a large validation set from the same source
X_tr, y_tr = torch.tensor(Xa, dtype=torch.float32), torch.tensor(ya)
X_va, y_va = torch.tensor(Xv, dtype=torch.float32), torch.tensor(yv)
print(f"train {tuple(X_tr.shape)}   validation {tuple(X_va.shape)}")

## 1. Overfitting, on purpose

**In plain English:** a model with far more knobs than data can learn the training examples by heart, noise
included, instead of the pattern behind them. It then looks perfect on its training data and worse on anything new.

A 2 → 64 → 64 → 2 MLP has about 4,500 parameters for 40 points. Train it full-batch with Adam and evaluate every
20 steps, keeping a copy of the weights with the best validation loss (we will need that for early stopping):

In [ ]:
def make_net(p_drop=0.0, width=64, seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, width), nn.ReLU(), nn.Dropout(p_drop),
                         nn.Linear(width, width), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(width, 2))

def evaluate(net, X, Y):
    net.eval()
    with torch.no_grad():
        logits = net(X)
    return F.cross_entropy(logits, Y).item(), (logits.argmax(1) == Y).float().mean().item()

def train(net, opt, steps=600, jitter=0.0, seed=0, every=20):
    gen = torch.Generator().manual_seed(seed)
    hist, best = {"step": [], "train": [], "val": []}, (math.inf, None)
    for step in range(steps):
        net.train()
        x = X_tr + jitter * torch.randn(X_tr.shape, generator=gen) if jitter else X_tr   # augmentation, if any
        opt.zero_grad()
        F.cross_entropy(net(x), y_tr).backward()
        opt.step()
        if step % every == 0 or step == steps - 1:
            val = evaluate(net, X_va, y_va)[0]
            hist["step"].append(step); hist["train"].append(evaluate(net, X_tr, y_tr)[0]); hist["val"].append(val)
            if val < best[0]:
                best = (val, copy.deepcopy(net.state_dict()))
    hist["best_state"] = best[1]
    return hist

base_net = make_net()
print(f"parameters: {sum(p.numel() for p in base_net.parameters()):,}")
base = train(base_net, torch.optim.Adam(base_net.parameters(), lr=1e-2))
tr_loss, tr_acc = evaluate(base_net, X_tr, y_tr)
va_loss, va_acc = evaluate(base_net, X_va, y_va)
print(f"train loss {tr_loss:.4f}  accuracy {tr_acc:.3f}")
print(f"val   loss {va_loss:.3f}  accuracy {va_acc:.3f}   (best val loss {min(base['val']):.3f} at step {base['step'][int(np.argmin(base['val']))]})")
assert tr_acc == 1.0 and va_acc < 0.9 and va_loss > 3 * min(base["val"])

In [ ]:
def plot_boundary(ax, net, title):
    xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    net.eval()
    with torch.no_grad():
        p = F.softmax(net(torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)), 1)[:, 1]
    ax.contourf(xx, yy, p.reshape(xx.shape).numpy(), levels=20, cmap="coolwarm", alpha=0.5)
    ax.contour(xx, yy, p.reshape(xx.shape).numpy(), levels=[0.5], colors="#0b0b0b", linewidths=1.2)
    ax.scatter(Xa[:, 0], Xa[:, 1], c=ya, cmap="coolwarm", edgecolor="#0b0b0b", s=22, linewidths=0.6)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(base["step"], base["train"], label="train loss")
ax1.plot(base["step"], base["val"], label="validation loss")
ax1.set(title="Train loss falls to ~0, validation loss turns and climbs", xlabel="step", ylabel="cross-entropy",
        yscale="log")
ax1.legend()
plot_boundary(ax2, base_net, "No regularisation: the boundary chases individual noisy points")
plt.tight_layout(); plt.show()

The signature of overfitting: **training loss keeps falling while validation loss bottoms out and rises**. Validation
*loss* rises much more than validation *accuracy* falls, because the model becomes extremely confident, and
confidently wrong predictions cost a lot of cross-entropy. The same picture, with eval loss on a held-out set, is how
you spot an over-trained LLM fine-tune (plus behavioural symptoms: verbatim training answers, rigid format).

## 2. Fix 1: weight decay, and why AdamW is not "Adam + L2"

**In plain English:** penalise large weights, so the network prefers smooth, simple functions unless the data insists.

With plain SGD, adding `λ/2·‖w‖²` to the loss (L2 regularisation) and shrinking the weights by `lr·λ·w` each step
(weight decay) are the same thing. Check:

In [ ]:
lam, lr = 0.1, 0.5
torch.manual_seed(0)
w_a = nn.Parameter(torch.randn(3)); w_b = nn.Parameter(w_a.detach().clone())
target = torch.tensor([1.0, -2.0, 0.5])
opt_a = torch.optim.SGD([w_a], lr=lr, weight_decay=lam)            # decay built into the optimizer
opt_b = torch.optim.SGD([w_b], lr=lr)                              # L2 penalty added to the loss
for _ in range(5):
    opt_a.zero_grad(); ((w_a - target) ** 2).sum().backward(); opt_a.step()
    opt_b.zero_grad(); (((w_b - target) ** 2).sum() + lam / 2 * (w_b ** 2).sum()).backward(); opt_b.step()
print("SGD weight_decay:", w_a.detach().numpy().round(5), "   SGD + L2 loss:", w_b.detach().numpy().round(5))
assert torch.allclose(w_a, w_b, atol=1e-6)

With Adam they are **not** the same. `Adam(weight_decay=λ)` adds `λ·w` to the gradient, and that sum is then divided by
`√v̂`: weights with large gradients get their decay divided away, weights with small gradients get decayed hard.
`AdamW` (Loshchilov & Hutter) *decouples* it: it shrinks every weight by `lr·λ·w` directly, outside the adaptive
step. Two weights of equal size, one with big gradients and one with tiny gradients:

In [ ]:
def decay_effect(opt_cls, steps=20, lam=0.1, lr=0.01):
    """How much weight decay moved each weight = final(with decay) - final(without)."""
    finals = []
    for wd in (lam, 0.0):
        w = nn.Parameter(torch.tensor([1.0, 1.0]))
        opt = opt_cls([w], lr=lr, weight_decay=wd)
        for _ in range(steps):
            w.grad = torch.tensor([10.0, 0.001])       # coordinate 0: big gradients; coordinate 1: tiny ones
            opt.step()
        finals.append(w.detach().clone())
    return (finals[1] - finals[0]).abs()                # extra shrinkage caused by decay

for opt_cls in (torch.optim.Adam, torch.optim.AdamW):
    d = decay_effect(opt_cls)
    print(f"{opt_cls.__name__:5s} extra shrink from decay: big-grad weight {d[0]:.5f}   tiny-grad weight {d[1]:.5f}")
adam_l2, adamw = decay_effect(torch.optim.Adam), decay_effect(torch.optim.AdamW)
assert adam_l2[1] > 100 * adam_l2[0]                 # L2 inside Adam: decay depends on gradient history
assert torch.allclose(adamw[0], adamw[1], rtol=0.01)  # AdamW: the same decay for every weight

AdamW decays both weights by the same amount, as weight decay should. Adam's L2 version barely touches the weight
with big gradients and decays the other roughly 1000× more, so the regulariser depends on gradient history rather
than on weight size. That is why AdamW is the default for transformers (typical `weight_decay` 0.01–0.1; biases and
norm weights are usually excluded). Now use it on the overfitting net, with a deliberately strong `weight_decay=1.0`
because 40 points is very little data:

In [ ]:
results = {"no regularisation": (base_net, base)}
wd_net = make_net()
results["weight decay (AdamW, 1.0)"] = (wd_net, train(wd_net, torch.optim.AdamW(wd_net.parameters(), lr=1e-2,
                                                                               weight_decay=1.0)))
for name, (net, h) in results.items():
    print(f"{name:28s} final val loss {h['val'][-1]:.3f}   val accuracy {evaluate(net, X_va, y_va)[1]:.3f}")

## 3. Fix 2: dropout, and why `model.eval()` matters

**In plain English:** during training, randomly switch off a fraction `p` of the neurons on every forward pass, so no
neuron can rely on a particular partner; the network learns redundant, more robust features. At inference, use all
of them.

PyTorch uses *inverted* dropout: in training mode the survivors are scaled by `1/(1−p)` so the expected output is
unchanged, and in eval mode dropout is the identity. The asserts check each claim:

In [ ]:
drop, p = nn.Dropout(p=0.5), 0.5
x = torch.ones(100_000)
torch.manual_seed(0)
drop.train()
out1, out2 = drop(x), drop(x)
drop.eval()
out_eval = drop(x)
print(f"train mode: {100 * (out1 == 0).float().mean():.1f}% zeros, survivors = {out1[out1 > 0].unique().tolist()}, "
      f"mean {out1.mean():.3f}, variance {out1.var():.3f}")
print(f"eval mode : zeros {int((out_eval == 0).sum())}, mean {out_eval.mean():.3f}, variance {out_eval.var():.3f}")
assert abs((out1 == 0).float().mean() - p) < 0.01 and torch.all(out1[out1 > 0] == 1 / (1 - p))
assert abs(out1.mean() - 1) < 0.01                    # expectation preserved by the 1/(1-p) scaling
assert abs(out1.var() - p / (1 - p)) < 0.02           # training mode adds variance p/(1-p) = 1.0 here
assert torch.equal(out_eval, x) and out_eval.var() == 0   # eval mode: identity, no noise
assert not torch.equal(out1, out2)                    # a different random mask on every training pass

The classic bug is forgetting `model.eval()` before validation or inference: predictions become random and slightly
worse, and two calls on the same input disagree. (Keeping dropout *on* at inference and averaging several passes is
a deliberate technique, "MC dropout", for a rough uncertainty estimate.) Train with dropout 0.3:

In [ ]:
do_net = make_net(p_drop=0.3)
results["dropout 0.3"] = (do_net, train(do_net, torch.optim.Adam(do_net.parameters(), lr=1e-2)))
print(f"dropout 0.3: final val loss {results['dropout 0.3'][1]['val'][-1]:.3f}   "
      f"val accuracy {evaluate(do_net, X_va, y_va)[1]:.3f}")

## 4. Fix 3 and 4: early stopping and data augmentation

**Early stopping** is free: keep the weights from the step with the best validation loss (our `train` already saved
them; see [02 · early stopping](02_training_loop_optimizers_lr_schedules.ipynb) for the patience logic).

**Data augmentation** makes more training data out of the data you have, using changes that should not change the
label. For images: flips, crops, colour jitter. For text: paraphrases, back-translation, synthetic examples from a
larger model. For our 2-D points: add a little Gaussian noise ("jitter") to every point on every step, which says
"points near a training point have the same label", exactly the smoothness we want.

In [ ]:
es_net = make_net()
es_net.load_state_dict(base["best_state"])            # the baseline run, rewound to its best validation step
results["early stopping"] = (es_net, dict(base, val=base["val"] + [min(base["val"])]))
aug_net = make_net()
results["augmentation (jitter 0.25)"] = (aug_net, train(aug_net, torch.optim.Adam(aug_net.parameters(), lr=1e-2),
                                                       jitter=0.25))

fig, axes = plt.subplots(1, 5, figsize=(11, 2.9))
summary = {}
for ax, (name, (net, h)) in zip(axes, results.items()):
    loss, acc = evaluate(net, X_va, y_va)
    summary[name] = (loss, acc)
    plot_boundary(ax, net, f"{name}\nval loss {loss:.2f}, acc {acc:.2f}")
plt.tight_layout(); plt.show()

print(f"{'':28s} {'val loss':>9s} {'val acc':>8s}")
for name, (loss, acc) in summary.items():
    print(f"{name:28s} {loss:9.3f} {acc:8.3f}")
base_loss, base_acc = summary["no regularisation"]
for name in list(summary)[1:]:
    assert summary[name][1] > base_acc, name                              # every fix improves val accuracy
for name in ["weight decay (AdamW, 1.0)", "early stopping", "augmentation (jitter 0.25)"]:
    assert summary[name][0] < base_loss / 2, name                         # and these also halve val loss

Every fix beats the unregularised net on validation accuracy, and the boundaries show why: they are smoother and
ignore the stray points. On this toy, the strongest are the ones that inject knowledge about the problem (augmentation) or
simply stop before memorisation starts (early stopping); dropout mostly improves accuracy while the model stays
over-confident. In practice you combine them, and you tune their strength on the validation set.

| Symptom | First things to try |
|---|---|
| train ≪ val loss, gap growing | more / augmented data, early stopping, weight decay, dropout, a smaller model |
| LLM fine-tune: eval loss rising after epoch 1–2 | fewer epochs, lower LR, LoRA instead of full fine-tuning, more diverse data |
| train and val both high | the opposite problem (underfitting): bigger model, longer training, higher LR, less regularisation |

## 5. Normalisation: BatchNorm vs LayerNorm

**In plain English:** as signals pass through many layers their scale drifts, which makes training unstable.
A normalisation layer rescales activations to mean 0 and variance 1, then applies a learned scale `γ` and shift
`β`. The two classic versions differ only in *which numbers they average over*:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
grid = np.random.default_rng(0).normal(size=(6, 8))
for ax, (title, rect) in zip(axes, [("BatchNorm: statistics per FEATURE, across the batch", (2.5, -0.5, 1, 6)),
                                    ("LayerNorm: statistics per EXAMPLE, across its features", (-0.5, 1.5, 8, 1))]):
    ax.imshow(grid, cmap="Greys", alpha=0.35)
    ax.add_patch(Rectangle(rect[:2], rect[2], rect[3], fill=False, ec="#eb6834", lw=3))
    ax.set(title=title, xlabel="feature (hidden dimension)", ylabel="example in the batch (or token)",
           xticks=range(8), yticks=range(6))
    ax.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
torch.manual_seed(0)
x = torch.randn(32, 8) * 3 + 5                               # batch of 32 examples, 8 features, mean 5, std 3
bn, ln = nn.BatchNorm1d(8), nn.LayerNorm(8)                  # gamma = 1, beta = 0 at init
bn_out, ln_out = bn(x), ln(x)

manual_bn = (x - x.mean(0)) / torch.sqrt(x.var(0, unbiased=False) + bn.eps)                   # over the batch axis
manual_ln = (x - x.mean(1, keepdim=True)) / torch.sqrt(x.var(1, unbiased=False, keepdim=True) + ln.eps)  # over features
assert torch.allclose(bn_out, manual_bn, atol=1e-5) and torch.allclose(ln_out, manual_ln, atol=1e-5)
print("BatchNorm: per-feature mean", bn_out.mean(0)[:4].detach().numpy().round(4), "std", bn_out.std(0, unbiased=False)[:4].detach().numpy().round(3))
print("LayerNorm: per-example mean", ln_out.mean(1)[:4].detach().numpy().round(4), "std", ln_out.std(1, unbiased=False)[:4].detach().numpy().round(3))

# the property that matters for sequence models: does example 0's output depend on the REST of the batch?
bn_alone_ish = bn(x[:4])[0]                                 # same example, different batch-mates
print("BatchNorm output for example 0 changes with the batch:", not torch.allclose(bn_out[0], bn_alone_ish, atol=1e-4))
print("LayerNorm output for example 0 changes with the batch:", not torch.allclose(ln_out[0], ln(x[:4])[0], atol=1e-6))
assert not torch.allclose(bn_out[0], bn_alone_ish, atol=1e-4) and torch.allclose(ln_out[0], ln(x[:4])[0], atol=1e-6)

bn.eval()                                                   # inference: BatchNorm switches to running averages
assert torch.allclose(bn(x)[0], bn(x[:4])[0])                # now batch-independent, but train and eval differ
print("running mean after 2 training batches:", bn.running_mean[:3].numpy().round(3), "(momentum 0.1 average of batch means)")

BatchNorm makes each example's output depend on its batch-mates, and it behaves differently in training (batch
statistics) and inference (running averages). For CNNs with big batches of fixed-size images that is fine, and the
batch noise even regularises a little. For language models it is a problem: sequences have different lengths
(padding pollutes the statistics), inference often runs one sequence at a time, and a model whose output depends on
what else is in the batch is hard to serve. LayerNorm normalises each token's vector on its own: no batch
dependence, identical in training and inference. In a transformer with activations `[batch, seq, d_model]`,
`nn.LayerNorm(d_model)` normalises every token independently over its `d_model` features.

## 6. RMSNorm from scratch (what Llama uses)

RMSNorm drops LayerNorm's mean-centring and bias: divide by the root-mean-square of the features, multiply by a
learned `weight`. One reduction instead of two and fewer parameters, with no measurable quality loss, so Llama,
Mistral, Qwen and most recent LLMs use it.

$$\mathrm{RMSNorm}(x) = \frac{x}{\sqrt{\tfrac{1}{d}\sum_i x_i^2 + \epsilon}} \odot w$$

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))          # learned scale; no bias

    def forward(self, x):
        rms_inv = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)   # 1 / sqrt(mean(x^2) + eps)
        return x * rms_inv * self.weight

torch.manual_seed(0)
h = torch.randn(2, 5, 16) * 4                              # [batch, seq, d_model]
ours, ref = RMSNorm(16), nn.RMSNorm(16, eps=1e-6)
with torch.no_grad():
    ours.weight.copy_(torch.rand(16)); ref.weight.copy_(ours.weight)
out = ours(h)
assert torch.allclose(out, ref(h), atol=1e-6)
rms_per_token = out.div(ours.weight).pow(2).mean(-1).sqrt()
print("matches torch.nn.RMSNorm:", torch.allclose(out, ref(h), atol=1e-6))
print("RMS of each normalised token (before the weight):", rms_per_token.flatten()[:5].detach().numpy().round(4))
assert torch.allclose(rms_per_token, torch.ones(2, 5), atol=1e-4)

centred = h - h.mean(-1, keepdim=True)                     # if the features already have mean 0 ...
ln_plain = nn.LayerNorm(16, eps=1e-6, elementwise_affine=False)
assert torch.allclose(RMSNorm(16)(centred), ln_plain(centred), atol=1e-5)
print("on zero-mean features RMSNorm == LayerNorm:", True)
print(f"parameters: LayerNorm(4096) {sum(p.numel() for p in nn.LayerNorm(4096).parameters()):,}   "
      f"RMSNorm(4096) {sum(p.numel() for p in RMSNorm(4096).parameters()):,}")

Implementation detail from real Llama code: the norm is computed in fp32 even when the model runs in bf16
(`x.float()`, normalise, cast back), because the mean of squares is exactly the kind of reduction that loses
precision in 16-bit.

| | BatchNorm | LayerNorm | RMSNorm |
|---|---|---|---|
| normalises over | the batch, per feature | the features, per example/token | the features, per token (no mean subtraction) |
| depends on other examples | yes | no | no |
| train vs inference behaviour | different (running stats) | identical | identical |
| learned parameters | γ, β (+ running stats) | γ, β | weight only |
| typical home | CNNs | BERT, GPT-2, original Transformer | Llama, Mistral, Qwen, most 2023+ LLMs |

## 7. Residual connections and gradient flow

A residual connection computes `x + f(x)` instead of `f(x)`. The derivative of `x + f(x)` with respect to `x` is
`I + f'(x)`: even if `f'` is tiny, the identity term passes the gradient straight through. Stack 30 layers and
compare the gradient norm reaching each layer:

In [ ]:
def layer_grad_norms(mode, depth=30, width=64):
    torch.manual_seed(0)
    lins = [nn.Linear(width, width) for _ in range(depth)]      # PyTorch default init everywhere
    norms = [nn.LayerNorm(width) for _ in range(depth)]
    h = torch.randn(256, width)
    for lin, norm in zip(lins, norms):
        if mode == "plain":
            h = torch.tanh(lin(h))                              # no residual
        elif mode == "pre-norm residual":
            h = h + torch.tanh(lin(norm(h)))                    # x + f(LN(x)): GPT-2, Llama
        else:
            h = norm(h + torch.tanh(lin(h)))                    # LN(x + f(x)): original Transformer, BERT
    h.pow(2).mean().backward()
    return np.array([lin.weight.grad.norm().item() for lin in lins])

flows = {m: layer_grad_norms(m) for m in ["plain", "post-norm residual", "pre-norm residual"]}
fig, ax = plt.subplots(figsize=(10, 4))
for name, g in flows.items():
    ax.semilogy(range(1, 31), g, marker="o", markersize=3, label=name)
ax.set(title="Gradient norm reaching each of 30 layers (layer 1 = next to the input)", xlabel="layer",
       ylabel="||dL/dW|| (log)")
ax.legend(); plt.show()
for name, g in flows.items():
    print(f"{name:20s} layer-1 / layer-30 gradient ratio {g[0] / g[-1]:.1e}")
assert flows["plain"][0] / flows["plain"][-1] < 1e-5
assert 0.3 < flows["pre-norm residual"][0] / flows["pre-norm residual"][-1] < 3
assert flows["post-norm residual"][0] / flows["post-norm residual"][-1] < flows["pre-norm residual"][0] / flows["pre-norm residual"][-1]

Without residuals the first layer gets a gradient about a million times smaller than the last. With pre-norm
residuals it is flat. Post-norm sits in between: the LayerNorm on the main path rescales the signal at every layer,
so gradients still shrink towards the input, which is why post-norm transformers need careful learning-rate warmup
and are hard to train very deep.

**The residual stream view (interpretability):** in a transformer, `x` is a running vector per token that every block
reads from (through its norm) and adds a small update to. Nothing overwrites it, which is why you can even delete a
middle layer of a big model and it mostly still works.

| | Post-norm `x = LN(x + f(x))` | Pre-norm `x = x + f(LN(x))` |
|---|---|---|
| used by | original Transformer (2017), BERT | GPT-2 onwards, Llama, Mistral, almost every modern LLM |
| residual path | passes through a LayerNorm every layer | a clean identity from embeddings to the last block |
| gradient to early layers | shrinks with depth (above) | roughly constant |
| warmup / depth | needs warmup, fragile when deep | tolerant, trains 100+ layers |
| extra piece | none | a final norm before the LM head (the residual stream is un-normalised) |

One pre-norm block, the part `tr02` asks about, is exactly: `x = x + attn(norm1(x))`, then `x = x + mlp(norm2(x))`.

## Try it yourself

**1.** Write LayerNorm from scratch with a learned `γ` (weight) and `β` (bias), and check it against
`nn.LayerNorm` after setting random `γ`, `β`.

In [ ]:
# Solution — try it yourself first, then run this
class MyLayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps, self.weight, self.bias = eps, nn.Parameter(torch.ones(d)), nn.Parameter(torch.zeros(d))

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, unbiased=False, keepdim=True)       # biased variance, like PyTorch
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias

mine, ref = MyLayerNorm(16), nn.LayerNorm(16)
with torch.no_grad():
    mine.weight.copy_(torch.randn(16)); mine.bias.copy_(torch.randn(16))
    ref.weight.copy_(mine.weight); ref.bias.copy_(mine.bias)
assert torch.allclose(mine(h), ref(h), atol=1e-5)
print("MyLayerNorm matches nn.LayerNorm, max diff:", (mine(h) - ref(h)).abs().max().item())

**2.** Sweep the dropout rate (0, 0.3, 0.6, 0.9) on the 40-point problem. Which gives the best validation
accuracy, and what happens at 0.9?

In [ ]:
# Solution — try it yourself first, then run this
sweep = {}
for p_drop in [0.0, 0.3, 0.6, 0.9]:
    net = make_net(p_drop=p_drop)
    train(net, torch.optim.Adam(net.parameters(), lr=1e-2), steps=400, every=400)
    sweep[p_drop] = (*evaluate(net, X_va, y_va), evaluate(net, X_tr, y_tr)[1])
    print(f"dropout {p_drop:.1f}: val loss {sweep[p_drop][0]:.3f}   val accuracy {sweep[p_drop][1]:.3f}   "
          f"train accuracy {sweep[p_drop][2]:.3f}")
assert sweep[0.3][1] > sweep[0.0][1] and sweep[0.9][2] < 1.0 and sweep[0.9][1] < sweep[0.3][1]
print("0.3 beats no dropout. At 0.9 the net can no longer fit even the training set (underfitting): its val loss")
print("looks good only because it is unconfident, while its val accuracy is the worst. Judge by the metric you care about.")

**3.** Show that `nn.BatchNorm1d` in training mode updates `running_mean` as an exponential moving average with
`momentum=0.1` (new = 0.9·old + 0.1·batch mean), and that in eval mode it does not update at all.

In [ ]:
# Solution — try it yourself first, then run this
bn3 = nn.BatchNorm1d(4)
xb = torch.randn(64, 4) + 3
before = bn3.running_mean.clone()
bn3.train(); bn3(xb)
assert torch.allclose(bn3.running_mean, 0.9 * before + 0.1 * xb.mean(0), atol=1e-6)
frozen = bn3.running_mean.clone()
bn3.eval(); bn3(xb + 100)
assert torch.equal(bn3.running_mean, frozen)
print("train mode: running_mean updated to", bn3.running_mean.numpy().round(3), "| eval mode: unchanged")

## Say it in an interview

- **Spotting overfitting:** training loss keeps falling while validation loss bottoms out and rises; the model is
  confident and wrong on new data. For an LLM fine-tune, also check behaviour on held-out prompts (verbatim
  training answers, format lock-in), not just eval loss.
- **Fixes, cheapest first:** early stopping (keep the best checkpoint), more or augmented data, weight decay,
  dropout, a smaller model; for LLMs: fewer epochs, lower LR, LoRA.
- **AdamW vs Adam + L2:** with SGD they are identical; with Adam, L2 goes through the `1/√v̂` normalisation so
  weights with large gradients are barely decayed. AdamW applies `w -= lr·λ·w` separately, the same for every weight.
- **Dropout:** zero a fraction `p` during training, scale survivors by `1/(1−p)`; identity at eval. Forgetting
  `model.eval()` gives noisy, non-deterministic predictions.
- **BatchNorm vs LayerNorm:** BN normalises each feature across the batch (batch-dependent, different at inference);
  LN normalises each token across its features (independent, identical at train and inference), which is why
  sequence models use it. **RMSNorm** = LayerNorm without mean-centring and bias: `x / rms(x) * w`, cheaper, used by
  Llama.
- **Residuals:** `d(x + f(x))/dx = I + f'(x)`, a gradient highway; pre-norm keeps that highway clean and trains deep
  stacks without delicate warmup; post-norm (original Transformer) shrinks gradients with depth.
- **Traps:** "BatchNorm in a transformer" (wrong for the reasons above); saying RMSNorm "normalises the variance"
  (it divides by the RMS, it does not subtract the mean); forgetting the final norm in a pre-norm stack; decaying
  norm weights and biases with weight decay.

## Next

- [04 · CNNs on digits](04_cnn_on_digits.ipynb): convolution, pooling, where BatchNorm lives.
- [05 · Embeddings, RNNs and the road to attention](05_embeddings_rnn_to_attention.ipynb): vanishing gradients
  through time.
- [Transformer block and tiny GPT](../07_genai_foundations/04_transformer_block_and_tiny_gpt.ipynb): pre-norm blocks
  with residuals, assembled · [Overfitting, regularization, cross-validation](../05_machine_learning/05_overfitting_regularization_cross_validation.ipynb)
  (the classic-ML view) · [Catastrophic forgetting](../11_finetuning/07_distillation_and_catastrophic_forgetting.ipynb)
- Theory: [dl_fundamentals course](../../dl_fundamentals/index.html) (chapters 7, 8 and 11) ·
  [interview bank](../../ai_interview_prep/index.html)